In [0]:
# ============================================================
# CHESS HISTORIC GAMES - GOLD LAYER CONFIGURATION
# ============================================================

SILVER_TABLE = "grandmasters.silver.historic_games"

# Gold tables - one per business analytics use case
GOLD_PLAYER_RETENTION    = "grandmasters.gold.player_retention_analysis"
GOLD_OPENING_TRENDS      = "grandmasters.gold.opening_meta_trends"
GOLD_MATCHMAKING_HEALTH  = "grandmasters.gold.matchmaking_health_index"
GOLD_ENGAGEMENT_TIMEZONE = "grandmasters.gold.engagement_target_timezones"
GOLD_GAME_SPEED          = "grandmasters.gold.game_speed_server_load"
GOLD_UPSET_RATE          = "grandmasters.gold.upset_rate_benchmark"

DEVELOPMENT_MODE = True

spark.sql("CREATE DATABASE IF NOT EXISTS grandmasters.gold")

print(f"Silver source: {SILVER_TABLE}")
print(f"Gold targets:")
print(f"  1. {GOLD_PLAYER_RETENTION}")
print(f"  2. {GOLD_OPENING_TRENDS}")
print(f"  3. {GOLD_MATCHMAKING_HEALTH}")
print(f"  4. {GOLD_ENGAGEMENT_TIMEZONE}")
print(f"  5. {GOLD_GAME_SPEED}")
print(f"  6. {GOLD_UPSET_RATE}")

In [0]:
# ============================================================
# READ SILVER
# ============================================================

silver_df = spark.table(SILVER_TABLE)

print(f"Total Silver Records: {silver_df.count()}")
print(f"Columns: {silver_df.columns}")
silver_df.printSchema()

display(silver_df.limit(10))

In [0]:
# ============================================================
# GOLD 1: PLATFORM PLAYER RETENTION ANALYSIS
#
# Business Question:
#   Are players who experience severe rating drops more likely
#   to stop playing? (Identifies churn risks)
#
# Pattern: Window Functions - SUM() OVER to calculate rolling
#   7-day net rating changes, tied to subsequent player inactivity.
# ============================================================

from pyspark.sql.functions import (
    col, sum as _sum, count as _count, max as _max, min as _min,
    avg as _avg, datediff, lit, when, explode, array, struct,
    row_number, lag, lead, round as _round
)
from pyspark.sql.window import Window

# Unpivot: create one row per player per game
white_games = (
    silver_df
    .select(
        col("white_player").alias("player_id"),
        col("utc_date"),
        col("white_elo").alias("rating"),
        col("white_rating_diff").alias("rating_diff"),
        col("player_tier"),
        col("game_id")
    )
)

black_games = (
    silver_df
    .select(
        col("black_player").alias("player_id"),
        col("utc_date"),
        col("black_elo").alias("rating"),
        col("black_rating_diff").alias("rating_diff"),
        col("player_tier"),
        col("game_id")
    )
)

player_games = white_games.unionByName(black_games).filter(col("player_id").isNotNull())

# Window: rolling 7-day rating change per player
player_window = Window.partitionBy("player_id").orderBy("utc_date")

retention_df = (
    player_games
    .groupBy("player_id", "utc_date", "player_tier")
    .agg(
        _sum("rating_diff").alias("daily_rating_change"),
        _count("game_id").alias("games_played"),
        _max("rating").alias("peak_rating")
    )
    # Days until next activity (inactivity gap)
    .withColumn("next_active_date", lead("utc_date").over(player_window))
    .withColumn("days_until_next_game", datediff("next_active_date", "utc_date"))
    # Rolling 7-game net rating change
    .withColumn("rolling_7game_rating_change",
        _sum("daily_rating_change").over(
            player_window.rowsBetween(-6, 0)
        )
    )
    # Churn risk flag: player inactive for 7+ days after rating drop
    .withColumn("churn_risk",
        when(
            (col("rolling_7game_rating_change") < -20) &
            (col("days_until_next_game") > 7), "high"
        ).when(
            (col("rolling_7game_rating_change") < -10) &
            (col("days_until_next_game") > 3), "medium"
        ).otherwise("low")
    )
    .drop("next_active_date")
)

print(f"Player Retention records: {retention_df.count()}")
retention_df.groupBy("churn_risk").count().show()
display(retention_df.limit(10))

In [0]:
# ============================================================
# GOLD 2: OPENING "META" TREND SHIFTS
#
# Business Question:
#   Which chess openings are rapidly growing or declining
#   in market share this quarter?
#
# Pattern: Quarter-over-Quarter (QoQ) Volume Growth Rate
#   comparison to see shifting platform trends.
# ============================================================

from pyspark.sql.functions import quarter, year, month, date_format, date_format

# Games per opening per month
opening_monthly = (
    silver_df
    .filter(col("eco").isNotNull())
    .withColumn("game_month", date_format("utc_date", "yyyy-MM"))
    .withColumn("game_quarter", quarter("utc_date"))
    .withColumn("game_year", year("utc_date"))
    .groupBy("eco", "opening", "game_month", "game_year", "game_quarter")
    .agg(
        _count("game_id").alias("games_count"),
        _avg("white_elo").alias("avg_player_elo")
    )
)

# Calculate month-over-month growth
month_window = Window.partitionBy("eco").orderBy("game_month")

opening_trends_df = (
    opening_monthly
    .withColumn("prev_month_count", lag("games_count").over(month_window))
    .withColumn("growth_rate_pct",
        _round(
            (col("games_count") - col("prev_month_count")) /
            col("prev_month_count") * 100, 2
        )
    )
    .withColumn("trend",
        when(col("growth_rate_pct") > 20, "rising")
        .when(col("growth_rate_pct") < -20, "declining")
        .otherwise("stable")
    )
    .drop("prev_month_count")
)

print(f"Opening Trends records: {opening_trends_df.count()}")
print("\nTop Rising Openings:")
opening_trends_df.filter(col("trend") == "rising").orderBy(col("growth_rate_pct").desc()).show(10, truncate=40)

display(opening_trends_df.limit(10))

In [0]:
# ============================================================
# GOLD 3: PLATFORM MATCHMAKING HEALTH INDEX
#
# Business Question:
#   Is our matchmaking system pairing players evenly,
#   or are Elo gaps too wide?
#
# Pattern: Statistical Distribution (AVG, STDDEV) of the
#   absolute Elo difference grouped by game type (Event).
# ============================================================

from pyspark.sql.functions import abs as _abs, stddev, percentile_approx

matchmaking_df = (
    silver_df
    .filter(col("white_elo").isNotNull() & col("black_elo").isNotNull())
    .withColumn("elo_difference", _abs(col("white_elo") - col("black_elo")))
    .groupBy("speed", "player_tier")
    .agg(
        _count("game_id").alias("total_games"),
        _round(_avg("elo_difference"), 1).alias("avg_elo_gap"),
        _round(stddev("elo_difference"), 1).alias("stddev_elo_gap"),
        _min("elo_difference").alias("min_elo_gap"),
        _max("elo_difference").alias("max_elo_gap"),
        percentile_approx("elo_difference", 0.5).alias("median_elo_gap"),
        percentile_approx("elo_difference", 0.95).alias("p95_elo_gap"),
        # Health indicator: % of games with Elo gap > 200
        _round(
            _sum(when(col("elo_difference") > 200, 1).otherwise(0)) /
            _count("game_id") * 100, 1
        ).alias("pct_wide_mismatch")
    )
    .withColumn("health_rating",
        when(col("avg_elo_gap") < 100, "excellent")
        .when(col("avg_elo_gap") < 200, "good")
        .when(col("avg_elo_gap") < 300, "fair")
        .otherwise("poor")
    )
)

print(f"Matchmaking Health records: {matchmaking_df.count()}")
display(matchmaking_df)

In [0]:
# ============================================================
# GOLD 4: MONETIZATION / ENGAGEMENT TARGET TIMEZONES
#
# Business Question:
#   When are our highest-rated (most dedicated) users active?
#   (Great for scheduling tournaments or maintenance)
#
# Pattern: Hourly Heatmap Aggregation matching average Elo
#   tiers against peak traffic hours of the day.
# ============================================================

from pyspark.sql.functions import date_format

# Create Elo tier buckets
engagement_df = (
    silver_df
    .filter(col("utc_hour").isNotNull() & col("white_elo").isNotNull())
    .withColumn("avg_game_elo", _round((col("white_elo") + col("black_elo")) / 2, 0))
    .withColumn("elo_tier",
        when(col("avg_game_elo") >= 2500, "2500+ (Elite)")
        .when(col("avg_game_elo") >= 2000, "2000-2499 (Expert)")
        .when(col("avg_game_elo") >= 1600, "1600-1999 (Intermediate)")
        .when(col("avg_game_elo") >= 1200, "1200-1599 (Club)")
        .otherwise("Below 1200 (Beginner)")
    )
    .withColumn("day_of_week", date_format("utc_date", "EEEE"))
    .groupBy("utc_hour", "elo_tier", "day_of_week")
    .agg(
        _count("game_id").alias("games_count"),
        _round(_avg("avg_game_elo"), 0).alias("avg_elo"),
        _round(_avg("white_elo"), 0).alias("avg_white_elo"),
        _round(_avg("black_elo"), 0).alias("avg_black_elo")
    )
    .orderBy("utc_hour", "elo_tier")
)

print(f"Engagement Timezone records: {engagement_df.count()}")
print("\nPeak hours by Elo tier (top 10 busiest slots):")
engagement_df.orderBy(col("games_count").desc()).show(10, truncate=False)

display(engagement_df.limit(20))

In [0]:
# ============================================================
# GOLD 5: GAME SPEED VS. SERVER LOAD METRICS
#
# Business Question:
#   Which time controls generate the highest volume of games
#   (and therefore the highest transaction overhead)?
#
# Pattern: Volume Grouping & Ranking of total games played
#   per day, categorized by strict base-time buckets.
# ============================================================

from pyspark.sql.functions import dense_rank

# Define time control buckets
game_speed_df = (
    silver_df
    .filter(col("time_control_initial").isNotNull())
    .withColumn("time_bucket",
        when(col("time_control_initial") <= 60, "UltraBullet (≤1min)")
        .when(col("time_control_initial") <= 180, "Bullet (1-3min)")
        .when(col("time_control_initial") <= 480, "Blitz (3-8min)")
        .when(col("time_control_initial") <= 900, "Rapid (8-15min)")
        .otherwise("Classical (15min+)")
    )
    .groupBy("utc_date", "time_bucket", "speed")
    .agg(
        _count("game_id").alias("daily_game_count"),
        _round(_avg("white_elo"), 0).alias("avg_player_rating"),
        _round(_avg(
            (col("time_control_initial") + col("time_control_increment") * 40) / 60
        ), 1).alias("avg_estimated_game_duration_min")
    )
)

# Rank time buckets by volume per day
rank_window = Window.partitionBy("utc_date").orderBy(col("daily_game_count").desc())

game_speed_df = (
    game_speed_df
    .withColumn("daily_volume_rank", dense_rank().over(rank_window))
    .orderBy("utc_date", "daily_volume_rank")
)

print(f"Game Speed records: {game_speed_df.count()}")
print("\nOverall volume by time bucket:")
game_speed_df.groupBy("time_bucket").agg(
    _sum("daily_game_count").alias("total_games")
).orderBy(col("total_games").desc()).show()

display(game_speed_df.limit(15))

In [0]:
# ============================================================
# GOLD 6: UPSET RATE BENCHMARK
#
# Business Question:
#   Do our platform's Elo calculations accurately reflect
#   actual win expectations across different tiers?
#
# Pattern: Conditional Aggregation calculating the percentage
#   of games where the lower-rated player won, segmented by
#   Elo brackets.
# ============================================================

# Determine who was favored and who won
upset_base = (
    silver_df
    .filter(
        col("white_elo").isNotNull() &
        col("black_elo").isNotNull() &
        col("result").isNotNull()  # exclude draws
    )
    .withColumn("elo_gap", _abs(col("white_elo") - col("black_elo")))
    .withColumn("elo_gap_bucket",
        when(col("elo_gap") < 50, "0-49 (Even)")
        .when(col("elo_gap") < 100, "50-99")
        .when(col("elo_gap") < 200, "100-199")
        .when(col("elo_gap") < 300, "200-299")
        .when(col("elo_gap") < 500, "300-499")
        .otherwise("500+ (Mismatch)")
    )
    .withColumn("higher_rated",
        when(col("white_elo") >= col("black_elo"), "white").otherwise("black")
    )
    .withColumn("is_upset",
        when(col("higher_rated") != col("result"), 1).otherwise(0)
    )
    .withColumn("avg_elo_bracket",
        when(((col("white_elo") + col("black_elo")) / 2) >= 2500, "2500+ Elite")
        .when(((col("white_elo") + col("black_elo")) / 2) >= 2000, "2000-2499 Expert")
        .when(((col("white_elo") + col("black_elo")) / 2) >= 1600, "1600-1999 Intermediate")
        .when(((col("white_elo") + col("black_elo")) / 2) >= 1200, "1200-1599 Club")
        .otherwise("Below 1200")
    )
)

upset_rate_df = (
    upset_base
    .groupBy("elo_gap_bucket", "avg_elo_bracket", "speed")
    .agg(
        _count("game_id").alias("total_games"),
        _sum("is_upset").alias("upset_count"),
        _round(_sum("is_upset") / _count("game_id") * 100, 2).alias("upset_rate_pct"),
        _round(_avg("elo_gap"), 0).alias("avg_elo_gap")
    )
    .orderBy("elo_gap_bucket", "avg_elo_bracket")
)

print(f"Upset Rate records: {upset_rate_df.count()}")
print("\nUpset Rate by Elo Gap:")
upset_rate_df.groupBy("elo_gap_bucket").agg(
    _sum("total_games").alias("games"),
    _round(_sum("upset_count") / _sum("total_games") * 100, 2).alias("overall_upset_pct")
).orderBy("elo_gap_bucket").show()

display(upset_rate_df)

In [0]:
# ============================================================
# WRITE ALL GOLD DELTA TABLES
# ============================================================

gold_tables = {
    GOLD_PLAYER_RETENTION: retention_df,
    GOLD_OPENING_TRENDS: opening_trends_df,
    GOLD_MATCHMAKING_HEALTH: matchmaking_df,
    GOLD_ENGAGEMENT_TIMEZONE: engagement_df,
    GOLD_GAME_SPEED: game_speed_df,
    GOLD_UPSET_RATE: upset_rate_df,
}

for table_name, df in gold_tables.items():
    if DEVELOPMENT_MODE:
        spark.sql(f"DROP TABLE IF EXISTS {table_name}")

    (
        df
        .write
        .format("delta")
        .mode("overwrite" if DEVELOPMENT_MODE else "append")
        .saveAsTable(table_name)
    )

    row_count = spark.table(table_name).count()
    print(f"✓ {table_name}: {row_count} rows")

print("\n" + "="*50)
print("ALL GOLD TABLES WRITTEN SUCCESSFULLY")